# Adaptive Transient Analysis

## Modeling

In [1]:
import xara

model = xara.Model(ndm=1, ndf=1)


Alternatively, the OpenSeesPy-compatible syntax is accepted:

> ```python
> model = xara.Model('basic', '-ndm', 1, '-ndf', 1)
> ```


Next the material is defined:


In [2]:
material = xara.UniaxialMaterial("Elastic", E=3000.0)
model.material(material)

where the OpenSeesPy-compatible syntax is:

> ```python
> material = 1
> model.uniaxialMaterial("Elastic", material, 3000.0)
> ```

In [3]:

section = xara.TrussSection("Truss", material, area=10.0)
model.section(section)

In [4]:
model.node(1,  0.0)
model.node(2, 72.0)

model.fix(1, 1);

In [5]:

model.element("Truss", 1, (1, 2), section=section); # 10.0, material)


## Loading

In [6]:
model.pattern("Plain", 1, "Linear")
model.load(2, 100.0)

## Analysis

In [7]:
model.constraints("Transformation")
model.numberer("ParallelPlain")
model.test("NormDispIncr", 1e-6, 6, 2)
model.system("ProfileSPD")
model.integrator("Newmark", 0.5, 0.25)
# model.analysis("Transient")
model.algorithm("Linear")
model.analysis("VariableTransient")


model.analyze(5, 0.0001, 0.00001, 0.001, 10)
time = model.getTime()

print(f"time: ", model.getTime())
approx_vtime = 0.0001 + 0.001  # One step at target, then one step at maximum
assert 0.99 < time / approx_vtime < 1.01,  (time,  approx_vtime)


model.setTime(0.0)
# Can still run a non-variable analysis by omitting the last three arguments:
model.analyze(5, 0.0001)
time = model.getTime()
print(f"time: ", model.getTime())
approx_vtime = 0.0001 * 5  # variable transient is not active so time should be dt * 5
# If variable transient is not active then time would be 0.0005
assert 0.99 < time / approx_vtime < 1.01,  (time,  approx_vtime)

time:  0.0011
time:  0.0005
